# Детекция автоматизированного трафика Авито

В этом ноутбуке я решаю задачу бинарной классификации для `cookie_id`. К положительному классу буду относить те объекты, чей трафик относится к известным сервисам
автоматизированного сбора данных. Основная метрика — **Precision при Recall ≥
70%**.

Финальный `submission.csv` получил **0.80397** при проверке на Stepik. Все
признаки я строю только по событиям внутри индивидуального суточного окна. Я не
использую внешние данные, API, большие языковые модели и сам `cookie_id` как
признак.


## Краткое описание подхода

1. **Подготовка данных.** Я проверяю границы окон и удаляю события раньше
   `window_start_ts` и не раньше `window_end_ts`. Это исключает временную утечку.
2. **Baseline.** Я обучаю одну HGB-модель на 13 простых агрегатах: количестве
   событий, длительности активности, числе уникальных объектов и возрасте cookie.
3. **Итоговые признаки.** Я добавляю статистики интервалов и сессий, признаки
   поиска и воронки действий, User-Agent, категории, последовательности событий,
   повторы с лагами и координаты указателя.
4. **Модель.** Я смешиваю процентильные ранги HGB и CatBoost. Один компонент
   CatBoost обучаю с временным затуханием, чтобы сильнее учитывать свежие данные.
5. **Валидация.** Я использую forward-срезы: каждая модель обучается только на
   прошлом и проверяется на следующих датах. Случайный split здесь не использую.

Архитектуру и веса ансамбля я выбираю по локальной временной валидации, после
чего один раз переобучаю зафиксированный pipeline на всём train.


## 1. Конфигурация и воспроизводимость

Я поддерживаю два расположения входных файлов (для того, чтобы проверяющий мог просто скачать архив репозитория, распаковать и завести всё из коробки, хоть изначально я и не хотел заливать датасеты на Github):

- приватный Kaggle Dataset `mihailivanovvvv/avito-entry-task-data`;
- папка `data` рядом с ноутбуком при запуске из репозитория.

Для `events` я поддерживаю как `events.csv`, так и исходный
`events.csv.gz`. Все random seed фиксирую, обучение выполняю на CPU.


In [2]:
import gc
import hashlib
import platform
import re
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import catboost
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

SEED = 42
CATBOOST_ALT_SEED = 2026
np.random.seed(SEED)

KAGGLE_ROOT = Path(
    "/kaggle/input/datasets/mihailivanovvvv/avito-entry-task-data"
)
LOCAL_ROOT = Path.cwd().resolve()


def resolve_input_paths() -> dict[str, Path]:
    """Находит одну из поддерживаемых структур данных без рекурсивного поиска."""
    candidate_roots = [KAGGLE_ROOT, LOCAL_ROOT, LOCAL_ROOT.parent]
    checked = []

    for root in dict.fromkeys(candidate_roots):
        data_dir = root / "data"
        train_path = data_dir / "train.csv"
        test_path = data_dir / "test.csv"
        event_candidates = [data_dir / "events.csv", data_dir / "events.csv.gz"]
        sample_candidates = [
            root / "sample_submission.csv",
            data_dir / "sample_submission.csv",
        ]

        events_path = next((path for path in event_candidates if path.is_file()), None)
        sample_path = next((path for path in sample_candidates if path.is_file()), None)
        checked.append(data_dir)

        if train_path.is_file() and test_path.is_file() and events_path and sample_path:
            return {
                "root": root,
                "data_dir": data_dir,
                "train": train_path,
                "test": test_path,
                "events": events_path,
                "sample": sample_path,
            }

    checked_text = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        "Не найдена поддерживаемая структура входных файлов. Проверены:\n"
        + checked_text
    )


INPUT_PATHS = resolve_input_paths()
INPUT_ROOT = INPUT_PATHS["root"]
DATA_DIR = INPUT_PATHS["data_dir"]
TRAIN_PATH = INPUT_PATHS["train"]
TEST_PATH = INPUT_PATHS["test"]
EVENTS_PATH = INPUT_PATHS["events"]
SAMPLE_PATH = INPUT_PATHS["sample"]

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = WORK_ROOT / "submission.csv"
VALIDATION_PATH = WORK_ROOT / "validation_metrics.csv"
REQUIREMENTS_PATH = WORK_ROOT / "requirements.txt"

print("Изпользованные версии библиотек и питона:")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("CatBoost:", catboost.__version__)


Изпользованные версии библиотек и питона:
Python: 3.12.13
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
NumPy: 2.0.2
pandas: 2.3.3
scikit-learn: 1.6.1
CatBoost: 1.2.10


In [3]:
# Явно проверяю все файлы, которые нужны для полного запуска.
required_paths = {
    "train.csv": TRAIN_PATH,
    "test.csv": TEST_PATH,
    EVENTS_PATH.name: EVENTS_PATH,
    "sample_submission.csv": SAMPLE_PATH,
}

missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Не найдены необходимые файлы:\n" + "\n".join(missing))

print("Источник данных:", INPUT_ROOT)
for name, path in required_paths.items():
    print(f"  {name}: {path}")
print("Итоговый файл:", OUTPUT_PATH)


Источник данных: /kaggle/input/datasets/mihailivanovvvv/avito-entry-task-data
  train.csv: /kaggle/input/datasets/mihailivanovvvv/avito-entry-task-data/data/train.csv
  test.csv: /kaggle/input/datasets/mihailivanovvvv/avito-entry-task-data/data/test.csv
  events.csv: /kaggle/input/datasets/mihailivanovvvv/avito-entry-task-data/data/events.csv
  sample_submission.csv: /kaggle/input/datasets/mihailivanovvvv/avito-entry-task-data/sample_submission.csv
Итоговый файл: /kaggle/working/submission.csv


## 2. Метрика

Для локальной оценки я реализую официальную метрику: максимальный precision
среди порогов, при которых recall не ниже 70%. Одинаковые `score` обрабатываю
одной группой, поэтому порядок строк с равными значениями не влияет на результат.


In [5]:
TARGET_RECALL = 0.70


def pr_curve(y_true, score) -> tuple[np.ndarray, np.ndarray]:
    """Возвращает precision и recall на границах групп равных score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")
    n_positive = int(y_true.sum())
    if n_positive == 0:
        return np.array([]), np.array([])
    order = np.argsort(-score, kind="mergesort")
    y_sorted, score_sorted = y_true[order], score[order]
    true_positive = np.cumsum(y_sorted)
    predicted_positive = np.arange(1, len(y_sorted) + 1)
    group_ends = np.r_[score_sorted[1:] != score_sorted[:-1], True]
    precision = true_positive[group_ends] / predicted_positive[group_ends]
    recall = true_positive[group_ends] / n_positive
    return precision, recall


def precision_at_recall(y_true, score, recall: float = TARGET_RECALL) -> float:
    precision, observed_recall = pr_curve(y_true, score)
    if len(precision) == 0:
        return float("nan")
    valid = observed_recall >= recall
    return float(precision[valid].max()) if valid.any() else 0.0


# Проверяю обычный случай и группу одинаковых score.
assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8
assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5
print("Тесты успешно пройдены")


Тесты успешно пройдены


## 3. Загрузка данных и контроль временных окон

Сначала я проверяю размер выборок, баланс классов и диапазоны дат. Затем для
каждой cookie оставляю только события из полуинтервала
`[window_start_ts, window_end_ts)`. События за пределами окна я не использую ни
в baseline, ни в итоговой модели.

В train и test я требую ровно одну строку на `cookie_id`. Точные повторы строк в
events не удаляю: отдельного идентификатора события в данных нет, поэтому
совпадающие действия в одну секунду могут быть реальными повторными событиями и
сами по себе полезны для детекции автоматизации. Для последовательностей я
сохраняю стабильный исходный порядок строк.

Пропуски в `item_id`, поиске и координатах зависят от типа события и являются
частью схемы данных. В числовых агрегатах отсутствие таких событий кодирую нулём,
а категориальные пропуски передаю CatBoost отдельным токеном `__MISSING__`.


In [6]:
DATE_COLUMNS = ["cookie_created_at", "window_start_ts", "window_end_ts"]


def load_data(data_dir: str | Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Загружает таблицы и сразу приводит все временные поля к datetime."""
    data_dir = Path(data_dir)
    train = pd.read_csv(data_dir / "train.csv", parse_dates=DATE_COLUMNS)
    test = pd.read_csv(data_dir / "test.csv", parse_dates=DATE_COLUMNS)
    events = pd.read_csv(EVENTS_PATH, parse_dates=["event_ts"])
    return train, test, events


# Загружаю данные и проверяю временные окна.
load_started = time.perf_counter()
train, test, events = load_data(DATA_DIR)

assert train["cookie_id"].is_unique
assert test["cookie_id"].is_unique
assert set(train["cookie_id"]).isdisjoint(set(test["cookie_id"]))
assert train["target"].isin([0, 1]).all()
assert events[["cookie_id", "event_ts", "event_name"]].notna().all().all()

exact_event_duplicates = int(events.duplicated().sum())

meta_audit = pd.concat(
    [
        train[["cookie_id", "window_start_ts", "window_end_ts"]].assign(split="train"),
        test[["cookie_id", "window_start_ts", "window_end_ts"]].assign(split="test"),
    ],
    ignore_index=True,
)
audit = events[["cookie_id", "event_ts"]].merge(
    meta_audit, on="cookie_id", how="inner", validate="many_to_one"
)
audit["before_window"] = audit["event_ts"] < audit["window_start_ts"]
audit["after_window"] = audit["event_ts"] >= audit["window_end_ts"]
audit_table = audit.groupby("split")[["before_window", "after_window"]].sum().astype(int)

print(f"train:  {train.shape}")
print(f"test:   {test.shape}")
print(f"events: {events.shape}")
print(f"Доля target=1: {train['target'].mean():.5f}")
print("Точные повторы строк events:", exact_event_duplicates)
print(
    "Период train:", train["window_start_ts"].min().date(),
    "—", train["window_start_ts"].max().date()
)
print(
    "Период test: ", test["window_start_ts"].min().date(),
    "—", test["window_start_ts"].max().date()
)
print("\nСобытия вне разрешённого окна — они будут исключены:")
print(audit_table.to_string())
print(f"\nЗагрузка и аудит: {time.perf_counter() - load_started:.1f} сек.")

del audit, meta_audit, exact_event_duplicates
_ = gc.collect()


train:  (11091, 5)
test:   (4909, 4)
events: (328905, 14)
Доля target=1: 0.08106
Точные повторы строк events: 4863
Период train: 2026-04-06 — 2026-04-19
Период test:  2026-04-20 — 2026-04-26

События вне разрешённого окна — они будут исключены:
       before_window  after_window
split                             
test               0             0
train              0         40779

Загрузка и аудит: 2.4 сек.


## 4. Baseline

В качестве понятной стартовой точки я использую одну
`HistGradientBoostingClassifier` с seed 42 и только 13 простыми числовыми
признаками:

- число событий, длительность активности и интенсивность;
- число уникальных типов событий, объявлений, категорий, локаций и запросов;
- максимальная страница поиска и число событий с координатами указателя;
- число платформ и User-Agent;
- возраст cookie к началу окна.

Я намеренно не добавляю в baseline категориальные сигнатуры, последовательности,
временное затухание и ансамблирование. Использую его чисто как неплохую базовую модель, которую буду стараться перебить.


In [7]:
def build_baseline_features(
    events: pd.DataFrame,
    meta: pd.DataFrame,
) -> pd.DataFrame:
    """Строит небольшой и интерпретируемый набор cookie-level агрегатов."""
    cookie_index = pd.Index(meta["cookie_id"], name="cookie_id")
    grouped = events.groupby("cookie_id", observed=True)
    features = pd.DataFrame(index=cookie_index)

    features["event_count"] = grouped.size()
    features["active_span_seconds"] = (
        grouped["event_ts"].max() - grouped["event_ts"].min()
    ).dt.total_seconds()
    features["unique_event_types"] = grouped["event_name"].nunique()
    features["unique_items"] = grouped["item_id"].nunique()
    features["unique_categories"] = grouped["item_category"].nunique()
    features["unique_locations"] = grouped["item_location"].nunique()
    features["unique_search_queries"] = grouped["search_query"].nunique()
    features["max_search_page"] = grouped["search_page"].max()
    features["pointer_event_count"] = grouped["pointer_x"].count()
    features["unique_platforms"] = grouped["platform"].nunique()
    features["unique_user_agents"] = grouped["user_agent"].nunique()
    features = features.fillna(0)

    # Интенсивность считаю устойчиво и для cookie с одним событием.
    active_minutes = np.maximum(features["active_span_seconds"] / 60.0, 1.0)
    features["events_per_minute"] = features["event_count"] / active_minutes

    indexed_meta = meta.set_index("cookie_id")
    cookie_age_days = (
        indexed_meta["window_start_ts"] - indexed_meta["cookie_created_at"]
    ).dt.total_seconds() / 86400.0
    features["cookie_age_days"] = (
        cookie_age_days.reindex(cookie_index).clip(lower=0).fillna(0)
    )

    return features.reset_index()


## 5. Итоговые поведенческие признаки

Автоматизированный сбор часто отличается не одним событием, а сочетанием
масштаба, регулярности и повторяемости. Поэтому для итоговой модели я строю
несколько групп признаков:

| Группа | Примеры | Мотивация |
|---|---|---|
| Активность | число событий, темп, длительность | Боты могут действовать быстрее и интенсивнее |
| Разнообразие | объявления, категории, локации | Сборщик часто охватывает больше объектов |
| Интервалы | квантили, регулярность, короткие паузы | Автоматические действия чаще имеют повторяемый ритм |
| Поиск и воронка | страницы, просмотры, контакты | Отличается путь от выдачи к объявлению и продавцу |
| Клиент | платформа и нормализованный User-Agent | Помогает выделить устойчивые технические профили |

Все агрегаты одной строки я считаю только по событиям соответствующей cookie.


In [8]:
ORIGINAL_EVENT_COLUMNS = [
    "cookie_id",
    "event_ts",
    "eid",
    "event_name",
    "platform",
    "user_agent",
    "item_id",
    "item_category",
    "item_location",
    "seller_type",
    "search_query",
    "search_page",
    "pointer_x",
    "pointer_y",
]


def _safe_token(value: object, max_prefix: int = 36) -> str:
    raw = str(value)
    slug = re.sub(r"[^0-9a-zA-Z]+", "_", raw.lower()).strip("_")[:max_prefix]
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:8]
    return f"{slug or 'value'}_{digest}"


def _put(out: pd.DataFrame, name: str, values: pd.Series, dtype: str = "float32") -> None:
    out[name] = values.reindex(out.index).fillna(0).astype(dtype)


def _distribution_summary(
    out: pd.DataFrame,
    ev: pd.DataFrame,
    col: str,
    prefix: str,
) -> None:
    valid = ev[["cookie_id", col]].dropna(subset=[col])
    if valid.empty:
        return
    freq = valid.groupby(["cookie_id", col], observed=True).size().rename("count").reset_index()
    totals = freq.groupby("cookie_id", observed=True)["count"].sum()
    distinct = freq.groupby("cookie_id", observed=True).size()
    maximum = freq.groupby("cookie_id", observed=True)["count"].max()
    freq["p"] = freq["count"] / freq["cookie_id"].map(totals)
    freq["entropy_part"] = -freq["p"] * np.log(freq["p"])
    freq["hhi_part"] = freq["p"] ** 2
    entropy = freq.groupby("cookie_id", observed=True)["entropy_part"].sum()
    hhi = freq.groupby("cookie_id", observed=True)["hhi_part"].sum()
    normalized_entropy = entropy / np.log(distinct.clip(lower=2))
    normalized_entropy = normalized_entropy.where(distinct > 1, 0)

    _put(out, f"{prefix}__nonmissing_count", totals)
    _put(out, f"{prefix}__nunique", distinct)
    _put(out, f"{prefix}__max_count", maximum)
    _put(out, f"{prefix}__max_share", maximum / totals)
    _put(out, f"{prefix}__entropy", entropy)
    _put(out, f"{prefix}__normalized_entropy", normalized_entropy)
    _put(out, f"{prefix}__hhi", hhi)


def _count_matrix(
    index: pd.Index,
    ev: pd.DataFrame,
    col: str,
    prefix: str,
    denominator: pd.Series,
    add_share: bool,
) -> pd.DataFrame:
    valid = ev[["cookie_id", col]].dropna(subset=[col])
    if valid.empty:
        return pd.DataFrame(index=index)
    counts = pd.crosstab(valid["cookie_id"], valid[col]).reindex(index, fill_value=0)
    columns: dict[str, pd.Series] = {}
    for value in sorted(counts.columns, key=lambda x: str(x)):
        token = _safe_token(value)
        series = counts[value].astype("float32")
        columns[f"{prefix}__count__{token}"] = series
        if add_share:
            columns[f"{prefix}__share__{token}"] = (
                series / denominator
            ).fillna(0).astype("float32")
    return pd.DataFrame(columns, index=index)


def _numeric_aggregates(out: pd.DataFrame, ev: pd.DataFrame, col: str, prefix: str) -> None:
    g = ev.groupby("cookie_id", observed=True)[col]
    for stat in ["count", "mean", "std", "min", "max", "median", "nunique"]:
        _put(out, f"{prefix}__{stat}", getattr(g, stat)())
    for quantile, label in [(0.10, "q10"), (0.25, "q25"), (0.75, "q75"), (0.90, "q90")]:
        _put(out, f"{prefix}__{label}", g.quantile(quantile))


def _max_bucket_count(ev: pd.DataFrame, bucket_col: str) -> pd.Series:
    return (
        ev.groupby(["cookie_id", bucket_col], observed=True)
        .size()
        .groupby("cookie_id", observed=True)
        .max()
    )


def prepare_in_window_events(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    if meta["cookie_id"].duplicated().any():
        raise ValueError("cookie_id must be unique in metadata")
    merged = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    in_window = (
        (merged["event_ts"] >= merged["window_start_ts"])
        & (merged["event_ts"] < merged["window_end_ts"])
    )
    merged = merged.loc[in_window].copy()
    if merged.empty:
        raise ValueError("No events remain after filtering by observation window")
    if not (
        (merged["event_ts"] >= merged["window_start_ts"])
        & (merged["event_ts"] < merged["window_end_ts"])
    ).all():
        raise AssertionError("Observation-window filter failed")
    return merged


def _add_user_agent_columns(ev: pd.DataFrame) -> None:
    ua = ev["user_agent"].fillna("").astype(str).str.lower()
    conditions = [
        ua.str.contains("python-requests", regex=False),
        ua.str.contains("python-urllib", regex=False),
        ua.str.contains("scrapy", regex=False),
        ua.str.contains("node-fetch", regex=False),
        ua.str.contains("curl/", regex=False),
        ua.str.contains("go-http-client", regex=False),
        ua.str.contains("headlesschrome", regex=False),
        ua.str.contains("yabrowser", regex=False),
        ua.str.contains("firefox", regex=False),
        ua.str.contains("avito/", regex=False),
        ua.str.contains("chrome", regex=False),
        ua.str.contains("safari", regex=False),
    ]
    labels = [
        "python_requests",
        "python_urllib",
        "scrapy",
        "node_fetch",
        "curl",
        "go_http_client",
        "headless_chrome",
        "yandex_browser",
        "firefox",
        "avito_app",
        "chrome",
        "safari",
    ]
    ev["ua_family"] = np.select(conditions, labels, default="other")

    os_conditions = [
        ua.str.contains("android", regex=False),
        ua.str.contains("iphone|ios", regex=True),
        ua.str.contains("windows", regex=False),
        ua.str.contains("macintosh|mac os", regex=True),
        ua.str.contains("linux", regex=False),
    ]
    ev["ua_os"] = np.select(
        os_conditions,
        ["android", "ios", "windows", "macos", "linux"],
        default="other",
    )
    ev["ua_is_automation"] = ua.str.contains(
        "headless|python-requests|python-urllib|scrapy|node-fetch|curl/|go-http-client",
        regex=True,
    )


def build_features(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Build cookie-level features using only events inside each cookie's window."""
    ev = prepare_in_window_events(events, meta)
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").reset_index(drop=True)
    ev["platform_norm"] = ev["platform"].astype(str).str.lower()
    _add_user_agent_columns(ev)

    ev["seconds_from_start"] = (ev["event_ts"] - ev["window_start_ts"]).dt.total_seconds()
    ev["second_bucket"] = ev["seconds_from_start"].astype("int64")
    ev["minute_bucket"] = (ev["seconds_from_start"] // 60).astype("int64")
    ev["hour_bucket"] = (ev["seconds_from_start"] // 3600).astype("int64")
    ev["clock_hour"] = ev["event_ts"].dt.hour.astype("int8")
    ev["delta_s"] = ev.groupby("cookie_id", observed=True)["event_ts"].diff().dt.total_seconds()
    ev["previous_event"] = ev.groupby("cookie_id", observed=True)["event_name"].shift()
    ev["event_transition"] = ev["previous_event"].fillna("__START__") + "__to__" + ev["event_name"]
    ev["same_event_as_previous"] = ev["event_name"].eq(ev["previous_event"])
    ev["same_type_delta_s"] = (
        ev.groupby(["cookie_id", "event_name"], observed=True)["event_ts"]
        .diff()
        .dt.total_seconds()
    )
    ev["previous_item_id"] = ev.groupby("cookie_id", observed=True)["item_id"].shift()
    ev["same_item_as_previous"] = ev["item_id"].notna() & ev["item_id"].eq(ev["previous_item_id"])
    for col in ["item_category", "item_location", "platform_norm", "ua_family"]:
        ev[f"previous_{col}"] = ev.groupby("cookie_id", observed=True)[col].shift()

    duplicate_subset = [col for col in ORIGINAL_EVENT_COLUMNS if col in ev.columns]
    ev["is_exact_duplicate"] = ev.duplicated(duplicate_subset, keep=False)

    index = pd.Index(meta["cookie_id"], name="cookie_id")
    out = pd.DataFrame(index=index)
    meta_indexed = meta.set_index("cookie_id").reindex(index)

    cookie_age_hours = (
        meta_indexed["window_start_ts"] - meta_indexed["cookie_created_at"]
    ).dt.total_seconds() / 3600
    out["meta__cookie_age_hours"] = cookie_age_hours.astype("float32")
    out["meta__log1p_cookie_age_hours"] = np.log1p(cookie_age_hours).astype("float32")
    out["meta__cookie_created_hour"] = meta_indexed["cookie_created_at"].dt.hour.astype("float32")
    out["meta__cookie_created_dow"] = meta_indexed["cookie_created_at"].dt.dayofweek.astype("float32")
    out["meta__window_dow"] = meta_indexed["window_start_ts"].dt.dayofweek.astype("float32")
    out["meta__window_is_weekend"] = (meta_indexed["window_start_ts"].dt.dayofweek >= 5).astype("float32")
    out["meta__new_cookie_1h"] = cookie_age_hours.le(1).astype("float32")
    out["meta__new_cookie_6h"] = cookie_age_hours.le(6).astype("float32")
    out["meta__new_cookie_24h"] = cookie_age_hours.le(24).astype("float32")
    out["meta__new_cookie_7d"] = cookie_age_hours.le(24 * 7).astype("float32")

    g = ev.groupby("cookie_id", observed=True)
    n_events = g.size().reindex(index).fillna(0).astype("float32")
    out["events__count"] = n_events
    _put(out, "events__timestamp_nunique", g["event_ts"].nunique())
    _put(out, "events__first_offset_s", g["seconds_from_start"].min())
    _put(out, "events__last_offset_s", g["seconds_from_start"].max())
    _put(out, "events__span_s", g["seconds_from_start"].max() - g["seconds_from_start"].min())
    _put(out, "events__active_seconds", g["second_bucket"].nunique())
    _put(out, "events__active_minutes", g["minute_bucket"].nunique())
    _put(out, "events__active_hours", g["hour_bucket"].nunique())
    _put(out, "events__max_per_second", _max_bucket_count(ev, "second_bucket"))
    _put(out, "events__max_per_minute", _max_bucket_count(ev, "minute_bucket"))
    _put(out, "events__max_per_hour", _max_bucket_count(ev, "hour_bucket"))
    _put(out, "events__exact_duplicate_count", g["is_exact_duplicate"].sum())
    _put(out, "events__exact_duplicate_rate", g["is_exact_duplicate"].mean())
    _put(out, "events__same_event_previous_rate", g["same_event_as_previous"].mean())
    _put(out, "events__same_item_previous_rate", g["same_item_as_previous"].mean())
    _put(out, "events__automation_ua_count", g["ua_is_automation"].sum())
    _put(out, "events__automation_ua_rate", g["ua_is_automation"].mean())

    out["events__per_active_minute"] = (
        out["events__count"] / out["events__active_minutes"].clip(lower=1)
    ).astype("float32")
    out["events__per_span_hour"] = (
        out["events__count"] / (out["events__span_s"].clip(lower=1) / 3600)
    ).astype("float32")
    out["events__timestamp_repeat_rate"] = (
        1 - out["events__timestamp_nunique"] / out["events__count"].clip(lower=1)
    ).astype("float32")

    for col in ["item_category", "item_location", "platform_norm", "ua_family"]:
        previous = f"previous_{col}"
        comparable = ev[col].notna() & ev[previous].notna()
        comparable_events = ev.loc[comparable]
        if not comparable_events.empty:
            changed = comparable_events[col].ne(comparable_events[previous])
            _put(
                out,
                f"sequence__{col}_change_rate",
                changed.groupby(comparable_events["cookie_id"], observed=True).mean(),
            )

    source_columns = [
        "item_id",
        "item_category",
        "item_location",
        "seller_type",
        "search_query",
        "search_page",
        "pointer_x",
        "pointer_y",
        "user_agent",
        "platform_norm",
        "event_name",
    ]
    for col in source_columns:
        missing = ev[col].isna().groupby(ev["cookie_id"], observed=True).sum()
        _put(out, f"missing__{col}__count", missing)
        out[f"missing__{col}__rate"] = (
            out[f"missing__{col}__count"] / out["events__count"].clip(lower=1)
        ).astype("float32")

    distribution_columns = [
        ("event_name", "event_name"),
        ("platform_norm", "platform"),
        ("user_agent", "user_agent"),
        ("ua_family", "ua_family"),
        ("ua_os", "ua_os"),
        ("item_id", "item"),
        ("item_category", "item_category"),
        ("item_location", "item_location"),
        ("seller_type", "seller_type"),
        ("search_query", "search_query"),
        ("search_page", "search_page"),
        ("event_transition", "transition_summary"),
    ]
    for col, prefix in distribution_columns:
        _distribution_summary(out, ev, col, prefix)

    count_columns = [
        ("event_name", "event_name", True),
        ("platform_norm", "platform", True),
        ("ua_family", "ua_family", True),
        ("ua_os", "ua_os", True),
        ("user_agent", "user_agent", False),
        ("item_category", "item_category", False),
        ("item_location", "item_location", False),
        ("seller_type", "seller_type", True),
        ("search_query", "search_query", False),
        ("search_page", "search_page_value", False),
        ("clock_hour", "clock_hour", True),
        ("event_transition", "transition", False),
    ]
    count_blocks = [
        _count_matrix(
            out.index,
            ev,
            col,
            prefix,
            out["events__count"].clip(lower=1),
            add_share,
        )
        for col, prefix, add_share in count_columns
    ]
    first_events = ev.groupby("cookie_id", observed=True).head(1)
    last_events = ev.groupby("cookie_id", observed=True).tail(1)
    unit_denominator = pd.Series(1.0, index=out.index)
    for boundary_name, boundary_frame in [("first", first_events), ("last", last_events)]:
        for col, prefix in [
            ("event_name", "event"),
            ("platform_norm", "platform"),
            ("ua_family", "ua_family"),
        ]:
            count_blocks.append(
                _count_matrix(
                    out.index,
                    boundary_frame,
                    col,
                    f"{boundary_name}_{prefix}",
                    unit_denominator,
                    False,
                )
            )
    out = pd.concat([out, *count_blocks], axis=1).copy()

    delta = ev.dropna(subset=["delta_s"]).copy()
    if not delta.empty:
        _numeric_aggregates(out, delta, "delta_s", "delta")
        dg = delta.groupby("cookie_id", observed=True)["delta_s"]
        for cutoff in [0, 1, 2, 5, 10, 30, 60, 300, 900, 1800]:
            _put(out, f"delta__le_{cutoff}s_rate", dg.apply(lambda x, c=cutoff: x.le(c).mean()))
        delta_freq = (
            delta.groupby(["cookie_id", "delta_s"], observed=True)
            .size()
            .rename("count")
            .reset_index()
        )
        _put(out, "delta__mode_count", delta_freq.groupby("cookie_id", observed=True)["count"].max())
        _put(out, "delta__nunique_exact", delta_freq.groupby("cookie_id", observed=True).size())
        out["delta__mode_share"] = (
            out["delta__mode_count"] / out["delta__count"].clip(lower=1)
        ).astype("float32")
        out["delta__cv"] = (
            out["delta__std"] / out["delta__mean"].clip(lower=1e-6)
        ).astype("float32")
        out["delta__iqr"] = (out["delta__q75"] - out["delta__q25"]).astype("float32")
        for divisor in [5, 10, 30, 60]:
            multiple_rate = dg.apply(
                lambda x, d=divisor: np.isclose(np.mod(x, d), 0).mean()
            )
            _put(out, f"delta__multiple_{divisor}s_rate", multiple_rate)
        for gap, label in [(300, "5m"), (1800, "30m"), (3600, "1h")]:
            sessions = dg.apply(lambda x, threshold=gap: 1 + x.gt(threshold).sum())
            _put(out, f"sessions__count_gap_{label}", sessions)

    for event_name in sorted(ev["event_name"].dropna().unique()):
        token = _safe_token(event_name)
        event_rows = ev.loc[ev["event_name"].eq(event_name)]
        event_group = event_rows.groupby("cookie_id", observed=True)
        for col, label in [
            ("item_id", "item"),
            ("item_category", "category"),
            ("item_location", "location"),
            ("search_query", "query"),
        ]:
            _put(
                out,
                f"event_specific__{token}__{label}_nunique",
                event_group[col].nunique(),
            )
        same_type_delta = event_rows.dropna(subset=["same_type_delta_s"])
        if not same_type_delta.empty:
            same_delta_group = same_type_delta.groupby("cookie_id", observed=True)["same_type_delta_s"]
            _put(out, f"event_specific__{token}__delta_median", same_delta_group.median())
            _put(out, f"event_specific__{token}__delta_q25", same_delta_group.quantile(0.25))
            _put(out, f"event_specific__{token}__delta_q75", same_delta_group.quantile(0.75))

    for col, prefix in [
        ("search_page", "search_page_numeric"),
        ("pointer_x", "pointer_x"),
        ("pointer_y", "pointer_y"),
    ]:
        _numeric_aggregates(out, ev, col, prefix)

    pointer = ev.dropna(subset=["pointer_x", "pointer_y"])
    if not pointer.empty:
        pointer_pairs = pointer.assign(
            pointer_pair=pointer["pointer_x"].astype(str) + "_" + pointer["pointer_y"].astype(str)
        ).copy()
        _distribution_summary(out, pointer_pairs, "pointer_pair", "pointer_pair")
        pg = pointer_pairs.groupby("cookie_id", observed=True)
        pointer_pairs["pointer_dt_s"] = pg["event_ts"].diff().dt.total_seconds()
        pointer_pairs["pointer_dx"] = pg["pointer_x"].diff()
        pointer_pairs["pointer_dy"] = pg["pointer_y"].diff()
        pointer_pairs["pointer_distance"] = np.hypot(
            pointer_pairs["pointer_dx"], pointer_pairs["pointer_dy"]
        )
        pointer_pairs["pointer_speed"] = (
            pointer_pairs["pointer_distance"]
            / pointer_pairs["pointer_dt_s"].where(pointer_pairs["pointer_dt_s"] > 0)
        )
        for col, prefix in [
            ("pointer_distance", "pointer_distance"),
            ("pointer_speed", "pointer_speed"),
            ("pointer_dx", "pointer_dx"),
            ("pointer_dy", "pointer_dy"),
        ]:
            _numeric_aggregates(out, pointer_pairs, col, prefix)
        valid_move = pointer_pairs["pointer_distance"].notna()
        if valid_move.any():
            move_frame = pointer_pairs.loc[valid_move]
            _put(
                out,
                "pointer_move__zero_distance_rate",
                move_frame["pointer_distance"].eq(0).groupby(move_frame["cookie_id"], observed=True).mean(),
            )
        out["pointer_x__range"] = (out["pointer_x__max"] - out["pointer_x__min"]).astype("float32")
        out["pointer_y__range"] = (out["pointer_y__max"] - out["pointer_y__min"]).astype("float32")
        out["pointer__bounding_box_area"] = (
            out["pointer_x__range"] * out["pointer_y__range"]
        ).astype("float32")

    search = ev.loc[ev["search_page"].notna()].copy()
    if not search.empty:
        search_group = search.groupby("cookie_id", observed=True)["search_page"]
        for threshold in [1, 2, 5, 10, 20]:
            _put(
                out,
                f"search_page__ge_{threshold}_rate",
                search_group.apply(lambda x, t=threshold: x.ge(t).mean()),
            )
        search["previous_search_page"] = search.groupby("cookie_id", observed=True)["search_page"].shift()
        search["search_page_delta"] = search["search_page"] - search["previous_search_page"]
        search_delta = search.dropna(subset=["search_page_delta"])
        if not search_delta.empty:
            _numeric_aggregates(out, search_delta, "search_page_delta", "search_page_delta")
            sdg = search_delta.groupby("cookie_id", observed=True)["search_page_delta"]
            _put(out, "search_page_delta__positive_rate", sdg.apply(lambda x: x.gt(0).mean()))
            _put(out, "search_page_delta__plus_one_rate", sdg.apply(lambda x: x.eq(1).mean()))

    query = ev.dropna(subset=["search_query"]).copy()
    if not query.empty:
        query["query_length"] = query["search_query"].astype(str).str.len()
        query["query_words"] = query["search_query"].astype(str).str.split().str.len()
        _numeric_aggregates(out, query, "query_length", "query_length")
        _numeric_aggregates(out, query, "query_words", "query_words")

    # Отношения отделяют структуру поведения от общего объёма трафика.
    ratio_pairs = [
        ("item__nunique", "item__nonmissing_count", "ratio__unique_items"),
        ("item_location__nunique", "item_location__nonmissing_count", "ratio__unique_locations"),
        ("item_category__nunique", "item_category__nonmissing_count", "ratio__unique_categories"),
        ("search_query__nunique", "search_query__nonmissing_count", "ratio__unique_queries"),
        ("search_page__nunique", "search_page__nonmissing_count", "ratio__unique_search_pages"),
        ("pointer_pair__nunique", "pointer_pair__nonmissing_count", "ratio__unique_pointer_pairs"),
    ]
    for numerator, denominator, name in ratio_pairs:
        if numerator in out and denominator in out:
            out[name] = (out[numerator] / out[denominator].clip(lower=1)).astype("float32")

    if "item_location__nunique" in out and "item__nunique" in out:
        out["ratio__locations_per_item"] = (
            out["item_location__nunique"] / out["item__nunique"].clip(lower=1)
        ).astype("float32")
    if "item_category__nunique" in out and "item__nunique" in out:
        out["ratio__categories_per_item"] = (
            out["item_category__nunique"] / out["item__nunique"].clip(lower=1)
        ).astype("float32")

    out = out.replace([np.inf, -np.inf], np.nan).fillna(0)
    out = out.loc[:, ~out.columns.duplicated()].astype("float32")
    if not np.isfinite(out.to_numpy()).all():
        raise ValueError("Feature matrix contains non-finite values")

    result = out.reset_index()
    if len(result) != len(meta) or result["cookie_id"].duplicated().any():
        raise AssertionError("Feature matrix has invalid cookie coverage")
    return result


def build_train_test_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
    events: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    meta = pd.concat(
        [
            train.drop(columns=["target"]).assign(_split="train"),
            test.assign(_split="test"),
        ],
        ignore_index=True,
    )
    combined = build_features(events, meta.drop(columns=["_split"]))
    split_by_cookie = meta.set_index("cookie_id")["_split"]
    combined["_split"] = combined["cookie_id"].map(split_by_cookie)
    train_features = combined.loc[combined["_split"].eq("train")].drop(columns="_split")
    test_features = combined.loc[combined["_split"].eq("test")].drop(columns="_split")
    train_features = train[["cookie_id"]].merge(
        train_features, on="cookie_id", how="left", validate="one_to_one"
    )
    test_features = test[["cookie_id"]].merge(
        test_features, on="cookie_id", how="left", validate="one_to_one"
    )
    if train_features.isna().any().any() or test_features.isna().any().any():
        raise AssertionError("Missing values appeared while aligning features")
    return train_features, test_features


### 5.1 Сессии и категориальные сигнатуры

Я добавляю короткие сессии и burst-серии, распределение активности внутри суток,
отношения между этапами воронки и компактные сигнатуры начала и конца истории.
Категориальные признаки передаю CatBoost напрямую; пропуски заменяю отдельным
значением `__MISSING__`, а не смешиваю с реальными категориями.


In [9]:
def _dominant_series(events: pd.DataFrame, column: str, index: pd.Index) -> pd.Series:
    valid = events[["cookie_id", column]].dropna().copy()
    valid[column] = valid[column].astype(str)
    counts = (
        valid.groupby(["cookie_id", column], observed=True)
        .size()
        .rename("count")
        .reset_index()
        .sort_values(
            ["cookie_id", "count", column],
            ascending=[True, False, True],
            kind="mergesort",
        )
    )
    dominant = counts.drop_duplicates("cookie_id").set_index("cookie_id")[column]
    return dominant.reindex(index).fillna("__MISSING__")


def _set_signature(
    events: pd.DataFrame,
    column: str,
    index: pd.Index,
    max_values: int = 8,
) -> pd.Series:
    unique = events[["cookie_id", column]].dropna().copy()
    unique[column] = unique[column].astype(str)
    unique = unique.drop_duplicates().sort_values(["cookie_id", column], kind="mergesort")
    distinct = unique.groupby("cookie_id", observed=True)[column].size()
    signature = unique.groupby("cookie_id", observed=True)[column].agg(" || ".join)
    signature = signature.where(distinct.le(max_values), "__MANY__")
    return signature.reindex(index).fillna("__MISSING__")


def _sequence_signature(
    events: pd.DataFrame,
    index: pd.Index,
    first: bool,
) -> pd.Series:
    selected = (
        events.groupby("cookie_id", observed=True).head(3)
        if first
        else events.groupby("cookie_id", observed=True).tail(3)
    )
    sequence = selected.groupby("cookie_id", observed=True)["event_name"].agg(
        lambda values: " > ".join(values.astype(str))
    )
    return sequence.reindex(index).fillna("__MISSING__")


def normalize_user_agent(value: str) -> str:
    value = str(value).lower()
    return re.sub(r"\d+(?:[._]\d+)*", "#", value)


def user_agent_family_major(value: str) -> str:
    value = str(value).lower()
    if "android" in value:
        os_name = "android"
    elif "iphone" in value or "ios" in value:
        os_name = "ios"
    elif "windows" in value:
        os_name = "windows"
    elif "macintosh" in value or "mac os" in value:
        os_name = "macos"
    elif "linux" in value:
        os_name = "linux"
    else:
        os_name = "other"

    patterns = [
        ("python_requests", r"python-requests/(\d+)"),
        ("headless", r"headlesschrome/(\d+)"),
        ("yandex", r"yabrowser/(\d+)"),
        ("firefox", r"firefox/(\d+)"),
        ("avito", r"avito/(\d+)"),
        ("chrome", r"chrome/(\d+)"),
        ("safari", r"version/(\d+)"),
    ]
    for family, pattern in patterns:
        match = re.search(pattern, value)
        if match:
            return f"{family}_{match.group(1)}_{os_name}"
    return f"other_{os_name}"


def build_cookie_categories(events: pd.DataFrame, cookie_ids: pd.Series) -> pd.DataFrame:
    ev = events.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
    ev["platform_clean"] = ev["platform"].fillna("__MISSING__").astype(str).str.lower()

    grouped = ev.groupby("cookie_id", observed=True)
    result = pd.DataFrame(index=pd.Index(cookie_ids, name="cookie_id"))

    mode_columns = [
        "user_agent",
        "platform_clean",
        "item_category",
        "item_location",
        "seller_type",
        "search_query",
    ]
    for column in mode_columns:
        result[f"cat__main_{column}"] = _dominant_series(ev, column, result.index)

    for column in ["user_agent", "platform_clean", "item_category", "seller_type"]:
        result[f"cat__set_{column}"] = _set_signature(ev, column, result.index)

    first = grouped.head(1).set_index("cookie_id")
    last = grouped.tail(1).set_index("cookie_id")
    for column in ["event_name", "user_agent", "platform_clean"]:
        result[f"cat__first_{column}"] = first[column].reindex(result.index).fillna("__MISSING__").astype(str)
        result[f"cat__last_{column}"] = last[column].reindex(result.index).fillna("__MISSING__").astype(str)

    main_ua = result["cat__main_user_agent"]
    result["cat__ua_normalized"] = main_ua.map(normalize_user_agent)
    result["cat__ua_family_major"] = main_ua.map(user_agent_family_major)

    # Сохраняю компактные сигнатуры начала и конца последовательности.
    result["cat__first3_events"] = _sequence_signature(ev, result.index, first=True)
    result["cat__last3_events"] = _sequence_signature(ev, result.index, first=False)

    return result.reset_index()


def _longest_run(mask: np.ndarray) -> int:
    if mask.size == 0:
        return 1
    best = current = 1
    for continues in mask:
        current = current + 1 if continues else 1
        if current > best:
            best = current
    return best


def _temporal_features(group: pd.DataFrame) -> pd.Series:
    seconds = group["seconds_from_start"].to_numpy(float)
    delta = np.diff(seconds)
    n = len(group)
    result: dict[str, float] = {}

    for cutoff, label in [
        (1, "1s"),
        (2, "2s"),
        (5, "5s"),
        (10, "10s"),
        (30, "30s"),
        (60, "1m"),
        (300, "5m"),
    ]:
        continues = delta <= cutoff
        result[f"v2__longest_burst_{label}"] = _longest_run(continues)
        session_starts = np.r_[True, ~continues]
        session_ids = np.cumsum(session_starts) - 1
        session_sizes = np.bincount(session_ids, minlength=int(session_ids.max()) + 1)
        result[f"v2__max_session_events_{label}"] = float(session_sizes.max())
        result[f"v2__mean_session_events_{label}"] = float(session_sizes.mean())

    if delta.size:
        result["v2__consecutive_delta_mae"] = float(np.mean(np.abs(np.diff(delta)))) if delta.size > 1 else 0.0
        result["v2__consecutive_delta_equal_rate"] = float(np.mean(np.diff(delta) == 0)) if delta.size > 1 else 0.0
        rounded = np.round(delta, 0)
        _, counts = np.unique(rounded, return_counts=True)
        probabilities = counts / counts.sum()
        result["v2__delta_rounded_entropy"] = float(-(probabilities * np.log(probabilities)).sum())
        result["v2__delta_top3_share"] = float(np.sort(counts)[-3:].sum() / counts.sum())
    else:
        result["v2__consecutive_delta_mae"] = 0.0
        result["v2__consecutive_delta_equal_rate"] = 0.0
        result["v2__delta_rounded_entropy"] = 0.0
        result["v2__delta_top3_share"] = 0.0

    # Считаю распределение активности внутри суточного окна.
    for hours, label in [(1 / 60, "first_minute"), (5 / 60, "first_5m"), (0.25, "first_15m"), (1, "first_hour")]:
        result[f"v2__{label}_share"] = float(np.mean(seconds < hours * 3600))
    for start_hour in [0, 6, 12, 18]:
        result[f"v2__window_q{start_hour // 6 + 1}_share"] = float(
            np.mean((seconds >= start_hour * 3600) & (seconds < (start_hour + 6) * 3600))
        )

    first_half = int(np.sum(seconds < 12 * 3600))
    second_half = n - first_half
    result["v2__second_to_first_half_ratio"] = second_half / max(first_half, 1)

    # Оцениваю равномерность движения по временной шкале.
    if n >= 3 and seconds[-1] > seconds[0]:
        x = np.arange(n, dtype=float)
        correlation = np.corrcoef(x, seconds)[0, 1]
        result["v2__timeline_linearity_r2"] = float(correlation * correlation) if np.isfinite(correlation) else 0.0
    else:
        result["v2__timeline_linearity_r2"] = 0.0

    return pd.Series(result, dtype="float32")


def build_v2_numeric_features(events: pd.DataFrame, cookie_ids: pd.Series) -> pd.DataFrame:
    ev = events.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
    ev["seconds_from_start"] = (ev["event_ts"] - ev["window_start_ts"]).dt.total_seconds()
    index = pd.Index(cookie_ids, name="cookie_id")

    temporal = ev.groupby("cookie_id", observed=True).apply(
        _temporal_features,
        include_groups=False,
    ).reindex(index).fillna(0)

    counts = pd.crosstab(ev["cookie_id"], ev["event_name"]).reindex(index, fill_value=0).astype(float)

    def event_count(name: str) -> pd.Series:
        if name in counts:
            return counts[name]
        return pd.Series(0.0, index=index)

    search = event_count("search_results_view")
    item = event_count("item_view")
    photo = event_count("photo_swipe")
    seller = event_count("seller_page_view")
    favorite = event_count("favorite_add")
    phone = event_count("contact_phone_show")
    chat = event_count("contact_chat_open")
    message = event_count("contact_message_sent")
    contact = phone + chat + message

    funnel = pd.DataFrame(index=index)
    denominator_pairs = {
        "v2__item_per_search": (item, search),
        "v2__photo_per_item": (photo, item),
        "v2__seller_per_item": (seller, item),
        "v2__favorite_per_item": (favorite, item),
        "v2__contact_per_item": (contact, item),
        "v2__message_per_chat": (message, chat),
        "v2__phone_per_item": (phone, item),
        "v2__engagement_per_item": (photo + seller + favorite + contact, item),
    }
    for name, (numerator, denominator) in denominator_pairs.items():
        funnel[name] = (numerator / denominator.clip(lower=1)).astype("float32")

    funnel["v2__contact_total"] = contact.astype("float32")
    funnel["v2__contact_type_nunique"] = pd.concat(
        [phone.gt(0), chat.gt(0), message.gt(0)], axis=1
    ).sum(axis=1).astype("float32")

    result = pd.concat([temporal, funnel], axis=1)
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    return result.reset_index()


### 5.2 Последовательности событий

Я не передаю модели сырые `item_id`. Вместо этого считаю статистики event
n-грамм, повторы с лагами, максимальные серии одинаковых действий, регулярность
интервалов, позиции капчи и контактов, проходы поисковых страниц и паттерны
координат. Такие признаки описывают сценарий поведения и меньше привязаны к
конкретным объявлениям.


In [10]:
MISSING = "__missing__"


def _ordered_events(events: pd.DataFrame) -> pd.DataFrame:
    ev = events.copy()
    ev["_row_order"] = np.arange(len(ev), dtype=np.int64)
    return ev.sort_values(
        ["cookie_id", "event_ts", "_row_order"],
        kind="mergesort",
    )




def _longest_equal_run(values: list[object]) -> int:
    if not values:
        return 0
    best = current = 1
    for previous, current_value in zip(values, values[1:]):
        if current_value == previous:
            current += 1
        else:
            current = 1
        best = max(best, current)
    return best


def _longest_true_run(values: np.ndarray) -> int:
    best = current = 0
    for value in values:
        current = current + 1 if bool(value) else 0
        best = max(best, current)
    return best


def _entropy(counter: Counter) -> float:
    total = sum(counter.values())
    if total == 0:
        return 0.0
    probabilities = np.fromiter(counter.values(), dtype=float) / total
    return float(-(probabilities * np.log(probabilities)).sum())


def _sequence_numeric(group: pd.DataFrame) -> pd.Series:
    event = group["event_name"].fillna(MISSING).astype(str).tolist()
    item = group["item_id"].fillna(-1).astype(str).tolist()
    category = group["item_category"].fillna(MISSING).astype(str).tolist()
    location = group["item_location"].fillna(MISSING).astype(str).tolist()
    query = group["search_query"].fillna(MISSING).astype(str).tolist()
    n = len(group)
    result: dict[str, float] = {}

    for size in (2, 3, 4):
        grams = [tuple(event[i : i + size]) for i in range(max(0, n - size + 1))]
        counts = Counter(grams)
        total = len(grams)
        result[f"v3__event_{size}gram_nunique"] = float(len(counts))
        result[f"v3__event_{size}gram_unique_rate"] = len(counts) / max(total, 1)
        result[f"v3__event_{size}gram_top_share"] = max(counts.values(), default=0) / max(total, 1)
        result[f"v3__event_{size}gram_entropy"] = _entropy(counts)

    for lag in (1, 2, 3, 4, 5):
        denominator = max(n - lag, 1)
        result[f"v3__event_lag{lag}_match_rate"] = (
            sum(a == b for a, b in zip(event[:-lag], event[lag:])) / denominator
            if n > lag else 0.0
        )
        result[f"v3__item_lag{lag}_match_rate"] = (
            sum(a == b and a != "-1" for a, b in zip(item[:-lag], item[lag:])) / denominator
            if n > lag else 0.0
        )

    result["v3__event_longest_run"] = float(_longest_equal_run(event))
    result["v3__item_longest_run"] = float(_longest_equal_run(item))
    result["v3__category_longest_run"] = float(_longest_equal_run(category))
    result["v3__location_longest_run"] = float(_longest_equal_run(location))
    result["v3__query_longest_run"] = float(_longest_equal_run(query))

    seconds = group["event_ts"].astype("int64").to_numpy(dtype=np.int64) / 1e9
    delta = np.diff(seconds)
    if delta.size:
        rounded = np.rint(delta).astype(np.int64)
        positive = rounded[rounded > 0]
        result["v3__delta_mad"] = float(np.median(np.abs(delta - np.median(delta))))
        result["v3__delta_lag1_corr"] = (
            float(np.corrcoef(delta[:-1], delta[1:])[0, 1])
            if delta.size >= 3 and np.std(delta[:-1]) > 0 and np.std(delta[1:]) > 0
            else 0.0
        )
        result["v3__delta_lag1_equal_rate"] = float(np.mean(rounded[:-1] == rounded[1:])) if delta.size > 1 else 0.0
        result["v3__delta_lag2_equal_rate"] = float(np.mean(rounded[:-2] == rounded[2:])) if delta.size > 2 else 0.0
        result["v3__delta_gcd"] = float(np.gcd.reduce(positive)) if positive.size else 0.0
        for divisor in (2, 5, 10, 15, 30, 60):
            result[f"v3__delta_multiple_{divisor}_rate"] = float(np.mean(positive % divisor == 0)) if positive.size else 0.0
    else:
        result["v3__delta_mad"] = 0.0
        result["v3__delta_lag1_corr"] = 0.0
        result["v3__delta_lag1_equal_rate"] = 0.0
        result["v3__delta_lag2_equal_rate"] = 0.0
        result["v3__delta_gcd"] = 0.0
        for divisor in (2, 5, 10, 15, 30, 60):
            result[f"v3__delta_multiple_{divisor}_rate"] = 0.0

    for event_name in ("captcha_shown", "login", "contact_phone_show", "contact_chat_open", "contact_message_sent"):
        positions = np.flatnonzero(group["event_name"].to_numpy() == event_name)
        prefix = f"v3__{event_name}"
        result[prefix + "_present"] = float(positions.size > 0)
        result[prefix + "_first_position_rate"] = float(positions[0] / max(n - 1, 1)) if positions.size else 1.0
        result[prefix + "_last_position_rate"] = float(positions[-1] / max(n - 1, 1)) if positions.size else 1.0
        result[prefix + "_after_first_share"] = float((n - positions[0] - 1) / n) if positions.size else 0.0

    pages = pd.to_numeric(group["search_page"], errors="coerce").dropna().to_numpy(float)
    if pages.size > 1:
        page_delta = np.diff(pages)
        result["v3__page_longest_plus_one_run"] = float(_longest_true_run(page_delta == 1) + 1)
        result["v3__page_longest_increasing_run"] = float(_longest_true_run(page_delta > 0) + 1)
        result["v3__page_reset_count"] = float(np.sum(page_delta < 0))
        result["v3__page_repeat_rate"] = float(np.mean(page_delta == 0))
    else:
        result["v3__page_longest_plus_one_run"] = float(pages.size)
        result["v3__page_longest_increasing_run"] = float(pages.size)
        result["v3__page_reset_count"] = 0.0
        result["v3__page_repeat_rate"] = 0.0

    pointer = group[["pointer_x", "pointer_y"]].dropna().astype(float)
    if len(pointer):
        x = pointer["pointer_x"].to_numpy()
        y = pointer["pointer_y"].to_numpy()
        for divisor in (5, 10, 25, 50):
            result[f"v3__pointer_grid_{divisor}_rate"] = float(np.mean((x % divisor == 0) & (y % divisor == 0)))
        result["v3__pointer_diagonal_rate"] = float(np.mean(x == y))
    else:
        for divisor in (5, 10, 25, 50):
            result[f"v3__pointer_grid_{divisor}_rate"] = 0.0
        result["v3__pointer_diagonal_rate"] = 0.0

    return pd.Series(result, dtype="float32")


def build_v3_numeric_features(
    events: pd.DataFrame,
    cookie_ids: pd.Series,
) -> pd.DataFrame:
    ev = _ordered_events(events)
    index = pd.Index(cookie_ids, name="cookie_id")
    result = (
        ev.groupby("cookie_id", observed=True)
        .apply(_sequence_numeric, include_groups=False)
        .reindex(index)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .astype("float32")
    )
    return result.reset_index()


def _ngram_signature(values: pd.Series, size: int, top_k: int = 3) -> str:
    tokens = values.fillna(MISSING).astype(str).tolist()
    grams = [">".join(tokens[i : i + size]) for i in range(max(0, len(tokens) - size + 1))]
    if not grams:
        return MISSING
    counts = Counter(grams)
    selected = sorted(counts.items(), key=lambda pair: (-pair[1], pair[0]))[:top_k]
    return " || ".join(gram for gram, _ in selected)


def _compressed_edge(values: pd.Series, first: bool, limit: int = 8) -> str:
    tokens = values.fillna(MISSING).astype(str).tolist()
    compressed = [token for i, token in enumerate(tokens) if i == 0 or token != tokens[i - 1]]
    selected = compressed[:limit] if first else compressed[-limit:]
    return ">".join(selected) if selected else MISSING


def build_v3_categories(
    events: pd.DataFrame,
    cookie_ids: pd.Series,
) -> pd.DataFrame:
    ev = _ordered_events(events)
    index = pd.Index(cookie_ids, name="cookie_id")
    grouped = ev.groupby("cookie_id", observed=True)
    result = pd.DataFrame(index=index)
    result["cat3__dominant_event_bigram"] = grouped["event_name"].agg(
        lambda values: _ngram_signature(values, 2, top_k=1)
    ).reindex(index)
    result["cat3__top3_event_bigrams"] = grouped["event_name"].agg(
        lambda values: _ngram_signature(values, 2, top_k=3)
    ).reindex(index)
    result["cat3__dominant_event_trigram"] = grouped["event_name"].agg(
        lambda values: _ngram_signature(values, 3, top_k=1)
    ).reindex(index)
    result["cat3__compressed_first8"] = grouped["event_name"].agg(
        lambda values: _compressed_edge(values, first=True)
    ).reindex(index)
    result["cat3__compressed_last8"] = grouped["event_name"].agg(
        lambda values: _compressed_edge(values, first=False)
    ).reindex(index)
    return result.fillna(MISSING).astype(str).reset_index()


## 6. Матрицы признаков

Для согласования схемы я строю признаки на общем списке cookie из train и test,
но `target` при этом не использую. После агрегации явно восстанавливаю порядок
через merge по `cookie_id`, проверяю пропуски и бесконечные значения. Baseline и
итоговые модели получают разные матрицы признаков.


In [11]:
# Строю признаки и разделяю их на матрицы моделей.
feature_started = time.perf_counter()

train_features, test_features = build_train_test_features(train, test, events)

meta = pd.concat([train.drop(columns="target"), test], ignore_index=True)
in_window_events = prepare_in_window_events(events, meta)
all_cookie_ids = meta["cookie_id"]

baseline_all = build_baseline_features(in_window_events, meta)

v2_numeric_all = build_v2_numeric_features(in_window_events, all_cookie_ids)
v2_categories_all = build_cookie_categories(in_window_events, all_cookie_ids)
v3_numeric_all = build_v3_numeric_features(in_window_events, all_cookie_ids)
v3_categories_all = build_v3_categories(in_window_events, all_cookie_ids)


def split_features(all_features: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_part = train[["cookie_id"]].merge(
        all_features, on="cookie_id", how="left", validate="one_to_one"
    )
    test_part = test[["cookie_id"]].merge(
        all_features, on="cookie_id", how="left", validate="one_to_one"
    )
    return train_part, test_part


baseline_train, _ = split_features(baseline_all)
v2_numeric_train, v2_numeric_test = split_features(v2_numeric_all)
v2_categories_train, v2_categories_test = split_features(v2_categories_all)
v3_numeric_train, v3_numeric_test = split_features(v3_numeric_all)
v3_categories_train, v3_categories_test = split_features(v3_categories_all)


def v1_feature_columns(columns: list[str]) -> list[str]:
    excluded_prefixes = ("search_query__count__", "transition__count__")
    return [
        column
        for column in columns
        if column != "cookie_id" and not column.startswith(excluded_prefixes)
    ]


X_baseline = baseline_train.drop(columns="cookie_id").copy()

base_columns = v1_feature_columns(train_features.columns.tolist())
cat_numeric_base_columns = [
    column for column in base_columns
    if not column.startswith("user_agent__count__")
]

X_hgb_v1 = train_features[base_columns].copy()
X_hgb_v1_test = test_features[base_columns].copy()

X_hgb_v2 = pd.concat(
    [X_hgb_v1, v2_numeric_train.drop(columns="cookie_id")], axis=1
)
X_hgb_v2_test = pd.concat(
    [X_hgb_v1_test, v2_numeric_test.drop(columns="cookie_id")], axis=1
)

X_cat_v2 = pd.concat(
    [
        train_features[cat_numeric_base_columns],
        v2_numeric_train.drop(columns="cookie_id"),
        v2_categories_train.drop(columns="cookie_id"),
    ],
    axis=1,
)
X_cat_v2_test = pd.concat(
    [
        test_features[cat_numeric_base_columns],
        v2_numeric_test.drop(columns="cookie_id"),
        v2_categories_test.drop(columns="cookie_id"),
    ],
    axis=1,
)

X_cat_v3 = pd.concat(
    [
        X_cat_v2,
        v3_numeric_train.drop(columns="cookie_id"),
        v3_categories_train.drop(columns="cookie_id"),
    ],
    axis=1,
)
X_cat_v3_test = pd.concat(
    [
        X_cat_v2_test,
        v3_numeric_test.drop(columns="cookie_id"),
        v3_categories_test.drop(columns="cookie_id"),
    ],
    axis=1,
)

cat_v2_columns = [column for column in X_cat_v2 if column.startswith("cat__")]
cat_v3_columns = [
    column for column in X_cat_v3
    if column.startswith(("cat__", "cat3__"))
]
y = train["target"].to_numpy(dtype=int)

numeric_frames = [
    X_baseline,
    X_hgb_v1,
    X_hgb_v1_test,
    X_hgb_v2,
    X_hgb_v2_test,
]
for frame in numeric_frames:
    assert not frame.isna().any().any()
    assert np.isfinite(frame.to_numpy()).all()

for frame, categorical in [
    (X_cat_v2, cat_v2_columns),
    (X_cat_v2_test, cat_v2_columns),
    (X_cat_v3, cat_v3_columns),
    (X_cat_v3_test, cat_v3_columns),
]:
    assert not frame[categorical].isna().any().any()
    numeric = frame.drop(columns=categorical)
    assert not numeric.isna().any().any()
    assert np.isfinite(numeric.to_numpy()).all()

print("Baseline HGB:", X_baseline.shape)
print("HGB, базовые признаки:", X_hgb_v1.shape)
print("HGB, расширенные признаки:", X_hgb_v2.shape)
print("CatBoost, базовые признаки:", X_cat_v2.shape, f"({len(cat_v2_columns)} категориальных)")
print("CatBoost, расширенные признаки:", X_cat_v3.shape, f"({len(cat_v3_columns)} категориальных)")
print(f"Построение всех признаков: {time.perf_counter() - feature_started:.1f} сек.")


del baseline_all, baseline_train
del v2_numeric_all, v2_categories_all, v3_numeric_all, v3_categories_all
del meta, in_window_events
_ = gc.collect()


Baseline HGB: (11091, 13)
HGB, базовые признаки: (11091, 787)
HGB, расширенные признаки: (11091, 832)
CatBoost, базовые признаки: (11091, 704) (20 категориальных)
CatBoost, расширенные признаки: (11091, 776) (25 категориальных)
Построение всех признаков: 179.3 сек.


## 7. Модели и финальный ансамбль

Baseline — одна небольшая HGB-модель. В финальном решении я смешиваю
процентильные ранги пяти более сильных компонентов:

- `0.30 × HGB` — средний ранг трёх seed;
- `0.40 × CatBoost`, seed 42, базовые признаки;
- `0.05 × CatBoost`, seed 2026, базовые признаки;
- `0.10 × CatBoost`, расширенные признаки;
- `0.15 × CatBoost`, расширенные признаки и временные веса.

Для последнего компонента использую вес `0.5 ** (age_days / 7)`: свежие
наблюдения сильнее влияют на модель, но старые не отбрасываются. Ранги смешиваю
вместо вероятностей, потому что итоговая метрика зависит от порядка объектов, а
калибровка разных моделей заметно различается.


In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

HGB_SEEDS = (42, 137, 2026)
CATBOOST_ITERATIONS = 650
TEMPORAL_HALF_LIFE_DAYS = 7.0

BLEND_WEIGHTS = {
    "hgb_v2": 0.30,
    "cat_v2_seed42": 0.40,
    "cat_v2_seed2026": 0.05,
    "cat_v3": 0.10,
    "cat_v3_temporal": 0.15,
}
assert np.isclose(sum(BLEND_WEIGHTS.values()), 1.0)

FORWARD_SPLITS = {
    "weekly_13_19": ("2026-04-13", "2026-04-20"),
    "block_11_13": ("2026-04-11", "2026-04-14"),
    "block_14_16": ("2026-04-14", "2026-04-17"),
    "block_17_19": ("2026-04-17", "2026-04-20"),
}



def make_baseline_model() -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_iter=200,
        max_leaf_nodes=7,
        min_samples_leaf=30,
        l2_regularization=2.0,
        early_stopping=False,
        random_state=SEED,
    )


def fit_baseline(
    X_fit: pd.DataFrame,
    y_fit: np.ndarray,
    X_predict: pd.DataFrame,
) -> np.ndarray:
    model = make_baseline_model()
    model.fit(X_fit, y_fit)
    probability = model.predict_proba(X_predict)[:, 1]
    del model
    _ = gc.collect()
    return probability

def percentile_rank(values: np.ndarray) -> np.ndarray:
    return pd.Series(values).rank(method="average", pct=True).to_numpy(dtype=float)


def temporal_decay_weights(dates: pd.Series, half_life_days: float) -> np.ndarray:
    dates = pd.to_datetime(dates)
    age_days = (dates.max() - dates).dt.total_seconds().to_numpy(dtype=float) / 86400.0
    weights = np.power(0.5, age_days / half_life_days)
    return weights / weights.mean()


def make_hgb(seed: int) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=500,
        max_leaf_nodes=15,
        min_samples_leaf=25,
        l2_regularization=2.0,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=seed,
    )


def fit_hgb_rank_ensemble(
    X_fit: pd.DataFrame,
    y_fit: np.ndarray,
    X_predict: pd.DataFrame,
) -> np.ndarray:
    ranked_predictions = []
    for seed in HGB_SEEDS:
        model = make_hgb(seed)
        model.fit(X_fit, y_fit)
        probability = model.predict_proba(X_predict)[:, 1]
        ranked_predictions.append(percentile_rank(probability))
        del model
    _ = gc.collect()
    return np.mean(ranked_predictions, axis=0)


def fit_catboost_rank(
    X_fit: pd.DataFrame,
    y_fit: np.ndarray,
    X_predict: pd.DataFrame,
    categorical_columns: list[str],
    seed: int,
    sample_weight: np.ndarray | None = None,
) -> np.ndarray:
    model = CatBoostClassifier(
        iterations=CATBOOST_ITERATIONS,
        depth=6,
        learning_rate=0.04,
        l2_leaf_reg=5.0,
        loss_function="Logloss",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        task_type="CPU",
    )
    model.fit(
        X_fit,
        y_fit,
        cat_features=categorical_columns,
        sample_weight=sample_weight,
    )
    probability = model.predict_proba(X_predict)[:, 1]
    del model
    _ = gc.collect()
    return percentile_rank(probability)


def blend_components(components: dict[str, np.ndarray]) -> np.ndarray:
    return sum(BLEND_WEIGHTS[name] * components[name] for name in BLEND_WEIGHTS)


def metric_row(split: str, name: str, y_true: np.ndarray, score: np.ndarray) -> dict:
    return {
        "split": split,
        "model": name,
        "precision_at_recall_0.70": precision_at_recall(y_true, score, recall=0.70),
        "pr_auc": average_precision_score(y_true, score),
        "roc_auc": roc_auc_score(y_true, score),
    }


## 8. Временная валидация

Train покрывает 6–19 апреля, а test — 20–26 апреля. Поэтому случайный split мог
бы завысить качество: соседние даты имеют похожие источники и паттерны трафика.
Я использую forward-validation — fit всегда расположен строго раньше validation.

Основным срезом считаю последнюю неделю 13–19 апреля. Три более коротких блока
использую как диагностику устойчивости по времени. Они частично пересекаются с
недельным срезом, поэтому среднее по четырём строкам — удобная сводка, а не
несмещённая оценка независимой кросс валидации.


In [13]:
# Валидирую модель только на будущих относительно fit датах.
validation_rows = []

for split_name, (valid_start, valid_end) in FORWARD_SPLITS.items():
    fit_mask = train["window_start_ts"].lt(valid_start).to_numpy()
    valid_mask = (
        train["window_start_ts"].ge(valid_start).to_numpy()
        & train["window_start_ts"].lt(valid_end).to_numpy()
    )

    assert fit_mask.any() and valid_mask.any()
    assert train.loc[fit_mask, "window_start_ts"].max() < train.loc[valid_mask, "window_start_ts"].min()

    print(
        f"{split_name}: fit={fit_mask.sum()}, valid={valid_mask.sum()}, "
        f"период={valid_start}—{valid_end}"
    )

    baseline_score = fit_baseline(
        X_baseline.loc[fit_mask],
        y[fit_mask],
        X_baseline.loc[valid_mask],
    )

    components = {}
    components["hgb_v2"] = fit_hgb_rank_ensemble(
        X_hgb_v2.loc[fit_mask],
        y[fit_mask],
        X_hgb_v2.loc[valid_mask],
    )
    components["cat_v2_seed42"] = fit_catboost_rank(
        X_cat_v2.loc[fit_mask],
        y[fit_mask],
        X_cat_v2.loc[valid_mask],
        cat_v2_columns,
        seed=SEED,
    )
    components["cat_v2_seed2026"] = fit_catboost_rank(
        X_cat_v2.loc[fit_mask],
        y[fit_mask],
        X_cat_v2.loc[valid_mask],
        cat_v2_columns,
        seed=CATBOOST_ALT_SEED,
    )
    components["cat_v3"] = fit_catboost_rank(
        X_cat_v3.loc[fit_mask],
        y[fit_mask],
        X_cat_v3.loc[valid_mask],
        cat_v3_columns,
        seed=SEED,
    )
    fit_temporal_weight = temporal_decay_weights(
        train.loc[fit_mask, "window_start_ts"], TEMPORAL_HALF_LIFE_DAYS
    )
    components["cat_v3_temporal"] = fit_catboost_rank(
        X_cat_v3.loc[fit_mask],
        y[fit_mask],
        X_cat_v3.loc[valid_mask],
        cat_v3_columns,
        seed=SEED,
        sample_weight=fit_temporal_weight,
    )

    reference_score = 0.30 * components["hgb_v2"] + 0.70 * components["cat_v2_seed42"]
    final_score = blend_components(components)

    for model_name, score in {
        "baseline_hgb": baseline_score,
        **components,
        "reference_blend": reference_score,
        "final_blend": final_score,
    }.items():
        validation_rows.append(metric_row(split_name, model_name, y[valid_mask], score))


validation_metrics = pd.DataFrame(validation_rows)
validation_metrics.to_csv(VALIDATION_PATH, index=False)

comparison = (
    validation_metrics[
        validation_metrics["model"].isin(
            ["baseline_hgb", "reference_blend", "final_blend"]
        )
    ]
    .pivot(
        index="split",
        columns="model",
        values="precision_at_recall_0.70",
    )
    .loc[list(FORWARD_SPLITS), ["baseline_hgb", "reference_blend", "final_blend"]]
)
comparison["gain_vs_baseline"] = comparison["final_blend"] - comparison["baseline_hgb"]
comparison["gain_vs_reference"] = comparison["final_blend"] - comparison["reference_blend"]

print("\nPrecision при Recall ≥ 70%:")
print(comparison.to_string(float_format=lambda value: f"{value:.4f}"))
print("\nСреднее по срезам:")
print(comparison.mean().to_string(float_format=lambda value: f"{value:.4f}"))


weekly_13_19: fit=5930, valid=5161, период=2026-04-13—2026-04-20
block_11_13: fit=4217, valid=2556, период=2026-04-11—2026-04-14
block_14_16: fit=6773, valid=2367, период=2026-04-14—2026-04-17
block_17_19: fit=9140, valid=1951, период=2026-04-17—2026-04-20

Precision при Recall ≥ 70%:
model         baseline_hgb  reference_blend  final_blend  gain_vs_baseline  gain_vs_reference
split                                                                                        
weekly_13_19        0.2834           0.7709       0.7751            0.4916             0.0042
block_11_13         0.2291           0.7015       0.7107            0.4815             0.0092
block_14_16         0.3107           0.7874       0.7931            0.4824             0.0057
block_17_19         0.2671           0.7943       0.7943            0.5272             0.0000

Среднее по срезам:
model
baseline_hgb        0.2726
reference_blend     0.7635
final_blend         0.7683
gain_vs_baseline    0.4957
gain_vs_referenc

## 9. Результаты экспериментов

### Локальная временная валидация

| Модель | Средний P@R≥70% | Что использую |
|---|---:|---|
| Простой HGB baseline | 0.2726 | 13 базовых агрегатов, один seed |
| Сильный reference | 0.7635 | `0.30 × HGB + 0.70 × CatBoost` |
| Финальный ансамбль | **0.7683** | последовательности, второй seed и temporal CatBoost |

Финальный ансамбль выигрывает у baseline на `0.4957`, а у уже сильного reference
на `0.0048`. Прирост относительно reference небольшой, но воспроизводится на
трёх из четырёх временных срезов и не ухудшает четвёртый.

### Что я пробовал

| Этап | Основное изменение | Результат платформы |
|---|---|---:|
| Первая версия | агрегаты событий и HGB/CatBoost | 0.76960 |
| Временные признаки | сессии, burst-серии, воронка и категории | 0.79607 |
| Настройка blend | фиксированные веса HGB и CatBoost | 0.79753 |
| Финальная версия | последовательности, второй seed и temporal weighting | **0.80397** |

Я также проверил TF-IDF с логистической регрессией на последовательностях
событий и User-Agent. Средний локальный `P@R70` был около `0.27`, а добавление
этой модели в ансамбль ухудшало результат, поэтому я её исключил. На платформу
я отправлял только зафиксированные версии pipeline, не подбирая ответы по
отдельным `cookie_id`.


## 10. Ограничения решения

- Разметка положительного класса основана на известных сервисах сбора данных.
  Я не ожидаю, что модель одинаково хорошо распознает принципиально новые типы
  автоматизации.
- Train охватывает только две недели, а доля положительного класса около 8%.
  Из-за этого метрика с ограничением на recall чувствительна к временному drift.
- Я работаю на уровне cookie и не использую IP, устройство или связи между
  cookie. Это ограничивает обнаружение распределённых ботов.
- User-Agent и поведенческие шаблоны можно имитировать; при изменении продукта
  часть признаков потребует пересмотра.
- Итоговый score — rank-blend. Он подходит для выбора порога, но не является
  откалиброванной вероятностью бота.
- Диагностические forward-срезы частично пересекаются. Поэтому я использую их
  для выбора устойчивого решения, а не как точную оценку будущего качества в
  production.


## 11. Финальное обучение и формирование submission

После выбора архитектуры я переобучаю все компоненты на полном train. Разметку
test нигде не использую. Перед сохранением проверяю порядок колонок, число строк,
уникальность `cookie_id`, отсутствие пропусков и диапазон `score`.

Эта ячейка является единственным местом, где формируется итоговый
`submission.csv`.


In [14]:
# Переобучаю выбранный ансамбль на полном train.
final_started = time.perf_counter()

final_components = {}

final_components["hgb_v2"] = fit_hgb_rank_ensemble(
    X_hgb_v2,
    y,
    X_hgb_v2_test,
)

final_components["cat_v2_seed42"] = fit_catboost_rank(
    X_cat_v2,
    y,
    X_cat_v2_test,
    cat_v2_columns,
    seed=SEED,
)

final_components["cat_v2_seed2026"] = fit_catboost_rank(
    X_cat_v2,
    y,
    X_cat_v2_test,
    cat_v2_columns,
    seed=CATBOOST_ALT_SEED,
)

final_components["cat_v3"] = fit_catboost_rank(
    X_cat_v3,
    y,
    X_cat_v3_test,
    cat_v3_columns,
    seed=SEED,
)

full_temporal_weight = temporal_decay_weights(
    train["window_start_ts"], TEMPORAL_HALF_LIFE_DAYS
)
final_components["cat_v3_temporal"] = fit_catboost_rank(
    X_cat_v3,
    y,
    X_cat_v3_test,
    cat_v3_columns,
    seed=SEED,
    sample_weight=full_temporal_weight,
)

test_score = blend_components(final_components)

prediction = pd.DataFrame({
    "cookie_id": test["cookie_id"],
    "score": test_score,
})

sample = pd.read_csv(SAMPLE_PATH)
assert sample["cookie_id"].is_unique
assert set(sample["cookie_id"]) == set(test["cookie_id"])

submission = sample[["cookie_id"]].merge(
    prediction[["cookie_id", "score"]],
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

assert submission.columns.tolist() == ["cookie_id", "score"]
assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert set(submission["cookie_id"]) == set(test["cookie_id"])
assert submission["score"].notna().all()
assert np.isfinite(submission["score"]).all()
assert submission["score"].between(0, 1).all()
assert submission["score"].nunique() > 1

submission.to_csv(OUTPUT_PATH, index=False)

requirements_text = (
    "numpy==2.0.2\n"
    "pandas==2.3.3\n"
    "scikit-learn==1.6.1\n"
    "catboost==1.2.10\n"
)
REQUIREMENTS_PATH.write_text(requirements_text, encoding="utf-8")


print(f"Сохранено: {OUTPUT_PATH}")
print("Строк:", len(submission))
print(f"Диапазон score: [{submission['score'].min():.6f}, {submission['score'].max():.6f}]")
print("Уникальных score:", submission["score"].nunique())
print("Дубликаты cookie_id:", int(submission["cookie_id"].duplicated().sum()))
print("Пропуски score:", int(submission["score"].isna().sum()))
print(f"Финальное обучение: {time.perf_counter() - final_started:.1f} сек.")

Сохранено: /kaggle/working/submission.csv
Строк: 4909
Диапазон score: [0.001395, 0.999582]
Уникальных score: 4845
Дубликаты cookie_id: 0
Пропуски score: 0
Финальное обучение: 308.8 сек.


## 12. Воспроизводимость и файлы результата

Для полного запуска нужны Python 3.12 и зависимости из автоматически создаваемого
`requirements.txt`. Интернет и GPU не требуются. При запуске **Restart & Run
All** ноутбук:

1. находит данные на Kaggle или в локальной папке `data`;
2. повторяет временную валидацию;
3. переобучает зафиксированный ансамбль на полном train;
4. сохраняет `submission.csv`, `validation_metrics.csv` и `requirements.txt`.

Перед отправкой я проверяю, что в `submission.csv` ровно **4 909 строк**, нет
дубликатов `cookie_id` и пропусков `score`. Файл через Excel не пересохраняю.
